# 🔍 RAG Lab — Exploration Notebook
**Day 9 · AI Application Development Bootcamp**

This notebook walks you through the RAG pipeline step by step, letting you **see the output at every stage** before writing a full application.

---

### How to use this notebook
- Run cells **in order** — later cells depend on earlier ones
- Cells marked `# ✏️ YOUR TURN` have gaps for you to fill in
- Cells marked `# 🔬 EXPERIMENT` invite you to change values and observe what happens
- Write your reflections in the **📝 Reflection** markdown cells

### Sections at a glance
| Part | Topic | Type |
|---|---|---|
| A | Guided pipeline walk-through | Run & read |
| B | Parameter experiments | Fill in + reflect |
| C | Open exploration (your own PDF, multi-query, scores) | Open |
| D | **Web-Augmented RAG** — live web search as a retriever | Fill in |
| E | **RAG Evaluation with RAGAS** *(optional)* — measure quality | Fill in |

### Before you start
Make sure your virtual environment is active and `GROQ_API_KEY` is set:
```bash
source .venv/bin/activate          # macOS/Linux
# or
.venv\Scripts\Activate.ps1        # Windows PowerShell

export GROQ_API_KEY="gsk_..."
```
Then launch Jupyter:
```bash
pip install jupyter  # if not already installed
jupyter notebook
```

---
## ⚙️ Setup — Install & Import

In [2]:
# Run this cell first — installs everything needed for the notebook
# (skip if you already ran pip install during the lecture setup)
import sys
!{sys.executable} -m pip install -q \
    groq python-dotenv \
    langchain langchain-community langchain-chroma \
    langchain-huggingface chromadb pypdf sentence-transformers \
    langchain-groq
print("✅ All packages ready")

✅ All packages ready


In [3]:
import os
import shutil
import math
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads GROQ_API_KEY from .env if present

# Quick check — will print ✅ if the key is available
api_key = os.getenv("GROQ_API_KEY")
if api_key:
    print(f"✅ GROQ_API_KEY found (starts with: {api_key[:8]}...)")
else:
    print("❌ GROQ_API_KEY not found — set it before continuing")
    print("   export GROQ_API_KEY='gsk_your_key_here'   # macOS/Linux")
    print("   $env:GROQ_API_KEY='gsk_your_key_here'    # Windows PS")

✅ GROQ_API_KEY found (starts with: gsk_iotO...)


---
## 🅐 PART A — Guided Pipeline Walk-Through

We will build the RAG pipeline one stage at a time and inspect the output at each step.  
Place any PDF you have in the same folder as this notebook and update `PDF_PATH` below.  
If you don't have one handy, you can save any Wikipedia article as a PDF from your browser.

### A1 — Step 1: Load the PDF

In [4]:
from langchain_community.document_loaders import PyPDFLoader

# 📌 Change this to your PDF file name
PDF_PATH = "sample.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"📄 Pages loaded: {len(pages)}")
print()
print("--- Metadata of the first page ---")
print(pages[0].metadata)
print()
print("--- First 500 characters of page 1 ---")
print(pages[0].page_content[:500])

📄 Pages loaded: 3

--- Metadata of the first page ---
{'producer': 'Skia/PDF m150', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36', 'creationdate': '2026-07-24T07:36:04+00:00', 'title': 'Great Sword - MH:World - Kiranico - Monster Hunter World: Iceborne Database', 'moddate': '2026-07-24T07:36:04+00:00', 'source': 'sample.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}

--- First 500 characters of page 1 ---
MONSTER HUNTER: WORLD / GUIDE
Great Sword
[!NOTE]
Please refer to the MHWI Damage Formula and MHWI General Data Sheet by MoonBunnie & Deathcream for up to date information.
The Great Sword (GS) is a slow, heavy weapon that sacrifices attack speed for attack strength. Learning how to use the GS effectively requires good
positioning, anticipation, and understanding of the GS's attack timings.
Strength & Weakness
Charging Attack
Combat Tips
Movelist
Motion Values
Strength & Weakness
Strength Weakne


**What you should see:**
- `pages` is a Python list — one `Document` object per page
- Each Document has a `metadata` dict containing at minimum `page` (0-indexed) and `source` (the filename)
- These metadata fields travel with each chunk all the way to ChromaDB and show up in your citations

> **Why does the page number start at 0?** PyPDFLoader uses 0-based indexing internally. When displaying to users, add 1: `page + 1`.

### A2 — Step 2: Split into Chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(pages)

print(f"✂️  Total chunks created: {len(chunks)}")
print(f"   Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print()
print("--- Sample: chunk #3 ---")
print(f"Metadata : {chunks[2].metadata}")
print(f"Length   : {len(chunks[2].page_content)} chars")
print(f"Content  : {chunks[2].page_content[:400]}")

✂️  Total chunks created: 10
   Average chunk size: 701 chars

--- Sample: chunk #3 ---
Metadata : {'producer': 'Skia/PDF m150', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36', 'creationdate': '2026-07-24T07:36:04+00:00', 'title': 'Great Sword - MH:World - Kiranico - Monster Hunter World: Iceborne Database', 'moddate': '2026-07-24T07:36:04+00:00', 'source': 'sample.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}
Length   : 470 chars
Content  : Left to right: Level 3 Charged Slash, Level 3 Strong Charged Slash, Level 3 True Charged Slash
If the player is hit, knocked away, evades, or stops the attack combo while charging a slash attack, the player will lose all the charge they've gained.
Combat Tips
Due to its slow speed and attacks, reckless play with the GS will expose the player to danger.
After an attack, the player can roll sideways


**Notice:**
- Chunk metadata still carries the original page number and source filename — inherited from the parent Document
- Chunk size varies because `RecursiveCharacterTextSplitter` respects natural break points
- `chunk_size=800` is the **maximum** — most chunks will be somewhat shorter

### A3 — Step 3: Load the Embedding Model

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

print("Loading embedding model... (first run downloads ~80 MB)")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Model loaded")

# Embed a single test sentence and inspect the vector
test_vector = embeddings.embed_query("What is the deadline for submission?")

print(f"\n🔢 Vector dimensions : {len(test_vector)}")
print(f"   First 8 values    : {[round(v, 4) for v in test_vector[:8]]}")
print(f"   Min value         : {min(test_vector):.4f}")
print(f"   Max value         : {max(test_vector):.4f}")

Loading embedding model... (first run downloads ~80 MB)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded

🔢 Vector dimensions : 384
   First 8 values    : [-0.0141, -0.0333, 0.0125, 0.0354, 0.0086, -0.0128, -0.1207, -0.0198]
   Min value         : -0.1540
   Max value         : 0.1974


**What you should see:**
- Vector has exactly **384 dimensions** (that's the output size of MiniLM-L6)
- Values are small floats, both positive and negative
- These 384 numbers together encode the *meaning* of the sentence

### A4 — Step 4: Demonstrate Cosine Similarity

Before storing anything, let's see cosine similarity in action — the mechanism that powers retrieval.

In [7]:
def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    dot   = sum(x * y for x, y in zip(a, b))
    mag_a = math.sqrt(sum(x ** 2 for x in a))
    mag_b = math.sqrt(sum(x ** 2 for x in b))
    return dot / (mag_a * mag_b)

sentences = {
    "query"    : "What is the late submission policy?",
    "similar"  : "Penalty for assignments handed in after the deadline",
    "related"  : "Students must attend all classes",
    "unrelated": "The best recipe for banana bread",
}

vectors   = {k: embeddings.embed_query(v) for k, v in sentences.items()}
query_vec = vectors["query"]

print(f'Base query: "{sentences["query"]}"\n')
for key in ["similar", "related", "unrelated"]:
    score = cosine_similarity(query_vec, vectors[key])
    bar   = "█" * int(score * 30)
    print(f"{score:.4f}  {bar}")
    print(f'         → "{sentences[key]}"')
    print()

Base query: "What is the late submission policy?"

0.4022  ████████████
         → "Penalty for assignments handed in after the deadline"

0.0741  ██
         → "Students must attend all classes"

-0.0138  
         → "The best recipe for banana bread"



**Expected pattern:** `similar` should score highest (different words, same meaning), `related` moderate, `unrelated` very low.  
This is exactly what ChromaDB does for every query — computed across all stored chunk vectors simultaneously.

### A5 — Step 5: Store in ChromaDB

In [8]:
from langchain_chroma import Chroma

CHROMA_DIR = "./chroma_lab_notebook"

if Path(CHROMA_DIR).exists():
    shutil.rmtree(CHROMA_DIR)
    print("🗑  Cleared old index")

print(f"Embedding {len(chunks)} chunks and storing in ChromaDB...")
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="lab_exploration",
)

count = vector_store._collection.count()
print(f"✅ Done — {count} vectors stored in ChromaDB")
print(f"   Index saved to: {CHROMA_DIR}/")

🗑  Cleared old index
Embedding 10 chunks and storing in ChromaDB...
✅ Done — 10 vectors stored in ChromaDB
   Index saved to: ./chroma_lab_notebook/


### A6 — Step 6: Retrieve & Inspect Results

In [9]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

QUESTION = "What are the main topics covered in this document?"

retrieved_docs = retriever.invoke(QUESTION)

print(f"❓ Question: {QUESTION}")
print(f"📋 Retrieved {len(retrieved_docs)} chunks:\n")

for i, doc in enumerate(retrieved_docs, 1):
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"--- Chunk {i} (page {page_display}) ---")
    print(doc.page_content[:300])
    print()

❓ Question: What are the main topics covered in this document?
📋 Retrieved 4 chunks:

--- Chunk 1 (page 1) ---
Left to right: Level 3 Charged Slash, Level 3 Strong Charged Slash, Level 3 True Charged Slash
If the player is hit, knocked away, evades, or stops the attack combo while charging a slash attack, the player will lose all the charge they've gained.
Combat Tips
Due to its slow speed and attacks, reckl

--- Chunk 2 (page 2) ---
Tackle (<<C>>) allows the player to brace for incoming attacks without knockback. Timing it in response to a monster's attack allows the player to
continue their Charged Slash combo without interruptions.
The Level 3 charged slash is performed by releasing <<T>> at the right time. Overcharging will 

--- Chunk 3 (page 2) ---
Movelist
In addition to the GS Charge Slashes above, the GS has a multitude of other attack chains. Mid-air attacks are done in the air, such as after hopping off a
ledge, jumping off a vine, or climbing up a wall and jumping off.
Move

**🔑 This is the most important debugging step in RAG.** Before looking at the LLM's answer, always inspect what the retriever found. If these chunks don't contain the answer, the LLM won't be able to answer correctly — no prompt tuning will fix a retrieval problem.

### A7 — Step 7: Generate an Answer with Groq

In [10]:
from groq import Groq

SYSTEM_PROMPT = """You are a document-based AI assistant.
Answer ONLY using the retrieved context provided below.
For every fact, include the page number in parentheses, e.g. (page 2).
If the answer is not in the context, say: I could not find that in the provided document."""

def format_context(docs):
    parts = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        parts.append(f"[Source {i} | page {page_display}]\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)

def ask_rag(question, retriever, model="llama-3.1-8b-instant", temperature=0.2):
    """Full RAG loop: retrieve → build prompt → generate → return (answer, docs)."""
    docs    = retriever.invoke(question)
    context = format_context(docs)
    prompt  = f"""Question: {question}

Retrieved context:
{context}

Answer using ONLY the context above. Include page references."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content, docs

answer, retrieved = ask_rag(QUESTION, retriever)

print(f"❓ Question: {QUESTION}")
print()
print("📋 Retrieved chunks:")
for i, doc in enumerate(retrieved, 1):
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  [{i}] page {page_display}: {doc.page_content[:80]}...")
print()
print("🤖 Answer:")
print(answer)

❓ Question: What are the main topics covered in this document?

📋 Retrieved chunks:
  [1] page 1: Left to right: Level 3 Charged Slash, Level 3 Strong Charged Slash, Level 3 True...
  [2] page 2: Tackle (<<C>>) allows the player to brace for incoming attacks without knockback...
  [3] page 2: Movelist
In addition to the GS Charge Slashes above, the GS has a multitude of o...
  [4] page 1: MONSTER HUNTER: WORLD / GUIDE
Great Sword
[!NOTE]
Please refer to the MHWI Damag...

🤖 Answer:
The main topics covered in this document are:

1. Combat Tips for using the Great Sword (GS) effectively (page 1).
2. Strength and Weakness of the GS, including its high damage output, decent reach, and slow movement and attack speed (page 1).
3. Charging Attack mechanics, including the importance of timing and the consequences of overcharging (page 2).
4. Movelist for the GS, including various attack chains and combos (page 2).
5. Tackle mechanics and its use in bracing for incoming attacks and continuing C

---
## 🅑 PART B — Experiments

Now you modify parameters and observe how they affect retrieval quality. **This is the core skill of RAG engineering.**

### B1 — ✏️ YOUR TURN: Chunk Size Experiment

Re-index your document with **three different chunk sizes** and compare how many chunks are created and how the retrieved text looks.  
Fill in the `build_index()` function below — it should work for all three sizes.

In [11]:
def build_index(pages, chunk_size, chunk_overlap, chroma_dir):
    """
    Build a ChromaDB index from pages with the given chunking parameters.
    Returns (vector_store, chunks)
    """
    # ✏️ YOUR TURN: create the splitter
    # Hint: RecursiveCharacterTextSplitter(chunk_size=..., chunk_overlap=..., separators=[...])
    splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", ". ", " ", ""],
)
    # ✏️ YOUR TURN: split the pages into chunks
    chunks = splitter.split_documents(pages)

    if Path(chroma_dir).exists():
        shutil.rmtree(chroma_dir)

    # ✏️ YOUR TURN: create the ChromaDB vector store
    # Hint: Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=chroma_dir, collection_name="exp")
    store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=chroma_dir, collection_name="exp")

    return store, chunks


# ── Run for three chunk sizes ─────────────────────────────────────────────────
test_question = "What are the main topics covered in this document?"
results = {}

for size in [300, 800, 1500]:
    overlap = int(size * 0.15)
    store, cks = build_index(pages, size, overlap, f"./chroma_exp_{size}")
    ret  = store.as_retriever(search_kwargs={"k": 3})
    docs = ret.invoke(test_question)
    results[size] = {"chunk_count": len(cks), "retrieved": docs}
    print(f"chunk_size={size:>4}: {len(cks):>4} chunks total | retrieved {len(docs)} docs")

print("\n✅ Experiment complete — run the next cell to compare retrieved text")

chunk_size= 300:   27 chunks total | retrieved 3 docs
chunk_size= 800:   10 chunks total | retrieved 3 docs
chunk_size=1500:    6 chunks total | retrieved 3 docs

✅ Experiment complete — run the next cell to compare retrieved text


In [12]:
# Inspect retrieved chunks for each size side-by-side
for size, data in results.items():
    print(f"\n{'='*60}")
    print(f"  chunk_size = {size}  ({data['chunk_count']} total chunks)")
    print(f"{'='*60}")
    for i, doc in enumerate(data["retrieved"], 1):
        page = doc.metadata.get('page', '?')
        page_display = page + 1 if isinstance(page, int) else page
        print(f"  [{i}] page {page_display} | {len(doc.page_content)} chars")
        print(f"      {doc.page_content[:200]}")
        print()


  chunk_size = 300  (27 total chunks)
  [1] page 1 | 288 chars
      [[images/greatsword_optimal)
Left to right: Level 3 Charged Slash, Level 3 Strong Charged Slash, Level 3 True Charged Slash
If the player is hit, knocked away, evades, or stops the attack combo while 

  [2] page 2 | 278 chars
      (strong) Charged Slash <<T>>
Tackle --
Brace for incoming attacks to minimize flinching and damage taken. This attack combos into:
(current) Charged Slash <<T>>
Jumping Wide Slash <<C>>
Side Blow R2+<

  [3] page 1 | 172 chars
      MONSTER HUNTER: WORLD / GUIDE
Great Sword
[!NOTE]
Please refer to the MHWI Damage Formula and MHWI General Data Sheet by MoonBunnie & Deathcream for up to date information.


  chunk_size = 800  (10 total chunks)
  [1] page 1 | 470 chars
      Left to right: Level 3 Charged Slash, Level 3 Strong Charged Slash, Level 3 True Charged Slash
If the player is hit, knocked away, evades, or stops the attack combo while charging a slash attack, the 

  [2] page 2 | 719

### 📝 Reflection B1

1. **How did chunk count change** as chunk_size increased from 300 → 800 → 1500?

   *Your answer:* The total chunks got smaller

2. **Which chunk size returned the most useful context** for your question? Why?

   *Your answer:* the 800 size was best in terms of balance and information. It got the gist of the pdf.

3. **With chunk_size=300**, did any retrieved chunk seem too short to be useful on its own?

   *Your answer:*  Yes.

4. **With chunk_size=1500**, did the retrieved chunk contain irrelevant sentences alongside the relevant ones?

   *Your answer:* Yes it did.

### B2 — ✏️ YOUR TURN: The k Parameter Experiment

Using a `chunk_size=800` index, compare what happens when you retrieve **k=1**, **k=4**, and **k=10** chunks.

In [13]:
store_800, _ = build_index(pages, 800, 120, "./chroma_k_exp")

# ✏️ YOUR TURN: pick a question relevant to YOUR document
question = "What are the main topics covered in this document?"  # Replace with something meaningful for your PDF

print(f"Question: {question}\n")

for k in [1, 4, 10]:
    # ✏️ YOUR TURN: create a retriever with this k value and retrieve docs
    # Hint: store_800.as_retriever(search_kwargs={"k": k})
    retriever_k = store_800.as_retriever(search_kwargs={"k": k})  # YOUR CODE HERE
    docs_k      = retriever_k.invoke(question)  # YOUR CODE HERE

    total_chars = sum(len(d.page_content) for d in docs_k)
    pages_found = [d.metadata.get('page', '?') for d in docs_k]
    print(f"k={k:2d} → {len(docs_k)} chunks | ~{total_chars} chars of context | pages: {pages_found}")

Question: What are the main topics covered in this document?

k= 1 → 1 chunks | ~470 chars of context | pages: [0]
k= 4 → 4 chunks | ~2622 chars of context | pages: [0, 1, 1, 0]
k=10 → 10 chunks | ~7013 chars of context | pages: [0, 1, 1, 0, 2, 0, 1, 1, 1, 2]


In [14]:
# Compare the ANSWERS generated with k=1 vs k=4 vs k=10
for k in [1, 4, 10]:
    ret_k    = store_800.as_retriever(search_kwargs={"k": k})
    ans_k, _ = ask_rag(question, ret_k)
    print(f"\n{'─'*60}")
    print(f"  k = {k}")
    print(f"{'─'*60}")
    print(ans_k)


────────────────────────────────────────────────────────────
  k = 1
────────────────────────────────────────────────────────────
The main topics covered in this document appear to be:

1. Slash attack mechanics (page 1)
2. Combat tips for the GS (page 1)

────────────────────────────────────────────────────────────
  k = 4
────────────────────────────────────────────────────────────
The main topics covered in this document are:

1. Combat Tips for the Great Sword (GS) (page 1, Source 1)
2. Tackle and its usage in combination with Charged Slash (page 2, Source 2)
3. Movelist for the GS, including various attack chains and combos (page 2, Source 3)
4. Strength and Weakness of the GS, including its high damage output, decent reach, and slow movement and attack speed (page 1, Source 4)

────────────────────────────────────────────────────────────
  k = 10
────────────────────────────────────────────────────────────
The main topics covered in this document are:

1. Combat Tips for using t

### 📝 Reflection B2

1. **k=1 vs k=4:** Did increasing k improve the answer? In what way?

   *Your answer:*Yes. k=4 provided more context and a more complete answer.

2. **k=10:** Better or worse than k=4? Any sign of context dilution?

   *Your answer:*Slightly worse. It included extra information that wasn't needed.

3. **If you had to pick one k value**, which would you choose and why?

   *Your answer:*k=4, because it gave enough context without too much irrelevant information.

### B3 — ✏️ YOUR TURN: Temperature Experiment

In [16]:
retriever_b3 = store_800.as_retriever(search_kwargs={"k": 4})
question_b3  = question  # reuse your question from B2

for temp in [0.0, 0.5, 1.0]:
    # ✏️ YOUR TURN: call ask_rag with the current temperature
    # Hint: ask_rag(question, retriever, temperature=temp)
    answer_t, _ = ask_rag(question_b3, retriever_b3, temperature=temp)  # YOUR CODE HERE

    print(f"\n{'─'*60}")
    print(f"  temperature = {temp}")
    print(f"{'─'*60}")
    print(answer_t)


────────────────────────────────────────────────────────────
  temperature = 0.0
────────────────────────────────────────────────────────────
The main topics covered in this document are:

1. Combat Tips for using the Great Sword (GS) effectively, including positioning, anticipation, and understanding of the GS's attack timings (Source 4, page 1).
2. The Great Sword's characteristics, such as its slow speed, heavy weight, and high damage output per swing (Source 4, page 1).
3. The GS's attack mechanics, including charging attacks, and the importance of timing to maximize damage (Source 2, page 2).
4. The GS's movelist, including various attack chains and combos, such as Charged Slashes, Side Blow, Rising Slash, Wide Slash, and Overhead Slash (Source 3, page 2).
5. Tips for using the GS in combat, such as rolling sideways or forward to recover quicker after an attack (Source 1, page 1).
6. The GS's strengths and weaknesses, including its decent reach, ability to block, and less depende

### 📝 Reflection B3

1. **temperature=0.0 vs 1.0:** How did the answer style differ? Was it more or less consistent on re-run?

   *Your answer:* Temperature 0.0 gave more consistent and focused answers. 1.0 was more varied between the re-runs.

2. **For a factual document Q&A system**, which temperature range would you recommend and why?

   *Your answer:* 0.0-0.3, because it gives more accurate and consistent factual answers and they dont differ too much on re-runs.

### B4 — 🔬 EXPERIMENT: Ask an Out-of-Scope Question

In [22]:
retriever_b4   = store_800.as_retriever(search_kwargs={"k": 4})
out_of_scope_q = "What is the current price of Bitcoin in Singapore dollars?"

answer_oos, docs_oos = ask_rag(out_of_scope_q, retriever_b4)

print(f"❓ Question: {out_of_scope_q}")
print()
print("📋 What the retriever found (likely unrelated):")
for doc in docs_oos:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  page {page_display}: {doc.page_content[:100]}...")
print()
print("🤖 Answer:")
print(answer_oos)

❓ Question: What is the current price of Bitcoin in Singapore dollars?

📋 What the retriever found (likely unrelated):
  page 1: Left to right: Level 3 Charged Slash, Level 3 Strong Charged Slash, Level 3 True Charged Slash
If th...
  page 3: <<T>> midair. When sliding stops, it turns into a normal Charged Slash.
Plunging Thrust Mid Air afte...
  page 2: Movelist
In addition to the GS Charge Slashes above, the GS has a multitude of other attack chains. ...
  page 1: MONSTER HUNTER: WORLD / GUIDE
Great Sword
[!NOTE]
Please refer to the MHWI Damage Formula and MHWI G...

🤖 Answer:
I could not find the current price of Bitcoin in Singapore dollars in the provided document.


### 📝 Reflection B4

1. Did the model correctly refuse, or did it hallucinate using irrelevant context?

   *Your answer:* It said that it cannot find the information on the PDF

2. If it hallucinated: what in `SYSTEM_PROMPT` could you change to make refusal more reliable?

   *Your answer:* it did not hallucinate

3. Modify `SYSTEM_PROMPT` in cell A7 to make refusal more explicit, re-run, and report what changed:

   *Your answer:*  The model correctly said the answer was not available

---
## 🅒 PART C — Open Exploration

### C1 — ✏️ YOUR TURN: Test Your Own PDF

In [25]:
# ✏️ YOUR TURN: load and index your own PDF, then ask 3 questions
MY_PDF = "sample.pdf"

# YOUR CODE HERE — follow the same steps as Parts A1 → A6
# Step 1: load   →  PyPDFLoader(MY_PDF).load()
pages = PyPDFLoader(MY_PDF).load()
# Step 2: chunk  →  build_index() from B1
chunk_size = 800
chunk_overlap = 120
store, chunks = build_index(
    pages,
    chunk_size,
    chunk_overlap,
    "./chroma_my_pdf2"
)
# Step 3: ask 3 questions with ask_rag()
retriever = store.as_retriever(search_kwargs={"k": 4})
print(ask_rag("What is the main topic of this document?", retriever)[0])
print()
print(ask_rag("What are the strengths and weaknesses of the Great Sword?", retriever)[0])
print()
print(ask_rag("How do charged slashes work?", retriever)[0])

The main topic of this document is the Great Sword (GS) in Monster Hunter: World, specifically its mechanics, combat tips, and movelist.

The strengths of the Great Sword are:

- High damage output per swing (Source 1)
- Decent Reach (Source 1)
- Can Block (Source 1)
- Less dependent on Sharpness maintenance (Source 1)

The weaknesses of the Great Sword are:

- Slow movement speed (Source 1, Source 2)
- Slow attack speed (Source 1, Source 2)
- Attack animations have long recovery times (Source 1, Source 2)

To answer your question about how charged slashes work, here's what I found in the context:

- A Charged Slash can be performed by holding <<T>> while performing a Standing/Walking attack (Source 2, page 2).
- After a Charged Slash, it can be comboed into various attacks, such as Side Blow <<T>>, Rising Slash <<T>>+<<C>>, and Wide Slash <<C>> (Source 2, page 2).
- A Charged Slash can be charged further by holding <<T>> (Source 2, page 2 and Source 3, page 2).
- If a Charged Slash is

### 📝 Reflection C1

| Question | Retrieval correct? | Answer quality (1–5) | Notes |
|---|---|---|---|
| Q1: What is the main topic of this document? | Yes | 5 | ... |
| Q2: What are the strengths and weaknesses of the Great Sword? | Yes | 5 | ... |
| Q3: How do charged slashes work? | Yes  | 5 | ... |

**Best chunk_size and k for your document?** *Your answer:* chunk_size = 800 k=4 is best.

### C2 — 🔬 EXPERIMENT: Multi-Query Retrieval

In [27]:
# from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY"),
)

base_retriever  = store_800.as_retriever(search_kwargs={"k": 4})
multi_retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=llm)

test_q        = question  # reuse your question from B2
docs_standard = base_retriever.invoke(test_q)
docs_multi    = multi_retriever.invoke(test_q)

print(f"Standard retrieval : {len(docs_standard)} chunks")
print(f"Multi-query        : {len(docs_multi)} chunks (after dedup)")

standard_texts = {d.page_content for d in docs_standard}
extra_chunks   = [d for d in docs_multi if d.page_content not in standard_texts]
print(f"\nExtra chunks found ONLY by multi-query: {len(extra_chunks)}")
for doc in extra_chunks:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    print(f"  [page {page_display}] {doc.page_content[:150]}")

Standard retrieval : 4 chunks
Multi-query        : 8 chunks (after dedup)

Extra chunks found ONLY by multi-query: 4
  [page 3] <<T>> midair. When sliding stops, it turns into a normal Charged Slash.
Plunging Thrust Mid Air after a Charged Rising Slash
> <<T>> Combo into a Stro
  [page 2] Strong Wide Slash <<C>>
True Charged Slash Back+<<T>>
True Charged
Slash
After a Strong Charged Slash >
Back+<<T>>
After a Tackle from a Strong
Charge
  [page 3] Move Button (+Condition) Properties
Tackle While charging a Strong Charged
Slash > <<C>> Upgrade a Strong Charged Slash to a True Charged Slash
Tackle
  [page 1] Slow movement speed
Slow attack speed
Attack animations have long recovery times
Charge Slashes
The Great Sword can empower its slashes by charging th


### C3 — 🔬 EXPERIMENT: Retrieval with Similarity Scores

In [28]:
results_with_scores = vector_store.similarity_search_with_score(question, k=6)

print(f"Question: {question}\n")
print(f"{'Similarity':>10}  {'Page':>5}  Content preview")
print("-" * 72)
for doc, score in results_with_scores:
    page = doc.metadata.get('page', '?')
    page_display = page + 1 if isinstance(page, int) else page
    similarity   = 1 / (1 + score)  # convert L2 distance to rough similarity
    print(f"{similarity:>10.4f}  {str(page_display):>5}  {doc.page_content[:60]}...")

Question: What are the main topics covered in this document?

Similarity   Page  Content preview
------------------------------------------------------------------------
    0.3773      1  Left to right: Level 3 Charged Slash, Level 3 Strong Charged...
    0.3639      2  Tackle (<<C>>) allows the player to brace for incoming attac...
    0.3513      2  Movelist
In addition to the GS Charge Slashes above, the GS ...
    0.3491      1  MONSTER HUNTER: WORLD / GUIDE
Great Sword
[!NOTE]
Please ref...
    0.3466      3  Move Button (+Condition) Properties
Tackle While charging a ...
    0.3439      1  Slow movement speed
Slow attack speed
Attack animations have...


### 📝 Reflection C — Overall

1. Is there a clear similarity score gap between the most and least relevant chunks?

   *Your answer:* No, the simiarities are close so no large gaps.

2. **Two most important RAG parameters to tune** for a new document collection?

   *Your answer:* Chunk size and k

3. **Which mini-project did you choose, and why?** What do you expect to be hardest?

   *Your answer:* 

---
## 🌐 PART D — Web-Augmented RAG

### What is Web-Augmented RAG?

So far, every answer has been grounded in a **local PDF** that you indexed yourself. But what if the user asks something that is not in any of your documents — for example, a current event, a recent product release, or a live regulation?

**Web-Augmented RAG** solves this by replacing (or combining) the vector-store retriever with a **live web search**. Instead of fetching chunks from ChromaDB, the system searches the web in real time, retrieves the top results, and uses those as the LLM's context.

```
Standard RAG:          Question → ChromaDB → chunks → LLM → answer
Web-Augmented RAG:     Question → Web search → live results → LLM → answer
Combined (best):       Question → ChromaDB + Web search → merge → LLM → answer
```

### Tool we use: Tavily Search

**Tavily** is a search API designed specifically for LLM applications. It returns clean, structured, LLM-friendly text from web results — not raw HTML. LangChain has a native `TavilySearchResults` integration.

**Free tier:** 1,000 searches/month — more than enough for this lab.  
**Sign up:** https://tavily.com → get your API key in 30 seconds.

### D0 — Install & Set Your Tavily Key

In [29]:
# Install the Tavily integration
import sys
!{sys.executable} -m pip install -q tavily-python langchain-community
print("✅ Tavily installed")

✅ Tavily installed


In [32]:
# Set your Tavily API key
# Get it from: https://app.tavily.com/home  (free, takes ~30 seconds)
#
# Option A — add to your .env file:    TAVILY_API_KEY=tvly-...
# Option B — set it here for this session only (do not commit this notebook with the key visible):
#
# import os
# os.environ["TAVILY_API_KEY"] = "tvly-your-key-here"
load_dotenv()  # reads TAVILY_API_KEY from .env if present

tavily_key = os.getenv("TAVILY_API_KEY") #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
if tavily_key:
    print(f"✅ TAVILY_API_KEY found (starts with: {tavily_key[:8]}...)")
else:
    print("❌ TAVILY_API_KEY not set.")
    print("   Get a free key at https://app.tavily.com/home")
    print("   Then add TAVILY_API_KEY=tvly-... to your .env file and re-run this cell.")

✅ TAVILY_API_KEY found (starts with: tvly-dev...)


### D1 — Guided: Web Search as a Retriever

Here we use `TavilySearchResults` as a drop-in retriever. The API returns a list of result objects, each with `content` (the page text) and `url`. We convert these into `Document` objects so the rest of our pipeline stays identical — the same `format_context()` and `ask_rag()` functions work unchanged.

In [33]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.documents import Document

# Create the web search tool
# max_results: how many web pages to retrieve per query (3–5 is a good range)
web_search = TavilySearchResults(
    max_results=4,
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
)

def web_search_to_docs(query: str) -> list:
    """
    Run a web search and convert results to Document objects
    so they work with our existing format_context() function.
    Each Document gets:
      - page_content: the retrieved text snippet
      - metadata["source"]: the URL
      - metadata["page"]: "web" (instead of a page number)
    """
    results = web_search.invoke(query)
    docs = []
    for r in results:
        docs.append(Document(
            page_content=r["content"],
            metadata={"source": r["url"], "page": "web"},
        ))
    return docs

# Test with a query that definitely requires current/live information
web_query = "Latest developments in AI regulation in Singapore 2025"
web_docs  = web_search_to_docs(web_query)

print(f"🌐 Web search: '{web_query}'")
print(f"   Retrieved {len(web_docs)} results\n")
for i, doc in enumerate(web_docs, 1):
    print(f"  [{i}] Source: {doc.metadata['source']}")
    print(f"       {doc.page_content[:200]}")
    print()

C:\Users\cheji\AppData\Local\Temp\ipykernel_1848\542086635.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(


🌐 Web search: 'Latest developments in AI regulation in Singapore 2025'
   Retrieved 4 results

  [1] Source: https://regulations.ai/regulations/RAI-SG-NA-SUMMARY-2026
       Singapore's AI regulatory landscape is dynamic and continuously evolving, with several key future developments anticipated. The Online Safety (Relief and Accountability) Bill (OSRA Bill) is currently 

  [2] Source: https://digital.nemko.com/regulations/singapore-ai-regulation
       ## 

## High-Impact and Generative AI Oversight

Singapore’s regulatory focus in 2025 extends to the governance of high-impact and generative AI systems, emphasizing robust safety testing, accountabil

  [3] Source: https://www.linkedin.com/posts/nicholasker_ai-regulations-in-2025-is-singapore-getting-activity-7310812823066984448-w7Hz
       AI Regulations in 2025: Is Singapore getting it right? AI is transforming industries at lightning speed, and with that comes a wave of evolving regulations. Hyperight’s latest article lays out six 

### D2 — Guided: Generate an Answer from Web Results

We can now use the same `format_context()` and the same Groq call — just swapping the source of the context from ChromaDB chunks to live web results.

In [34]:
WEB_SYSTEM_PROMPT = """You are a research assistant answering questions using live web search results.
Answer using ONLY the retrieved web content provided below.
For every fact, cite the source URL in parentheses.
If the information is not in the provided results, say: I could not find that in the retrieved web content.
Do NOT use your own training knowledge."""

def ask_web_rag(question: str, max_results: int = 4) -> str:
    """Full Web-RAG loop: web search → format context → Groq → answer."""
    # Step 1: retrieve from the web
    docs    = web_search_to_docs(question)
    context = format_context(docs)  # reusing the same function from Part A

    # Step 2: build prompt and generate
    prompt = f"""Question: {question}

Retrieved web content:
{context}

Answer using ONLY the retrieved content above. Cite the source URL for every claim."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": WEB_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# Ask something that requires live, current information
live_question = "What are the latest AI safety regulations announced in 2025?"

print(f"❓ Question: {live_question}")
print()
answer_web = ask_web_rag(live_question)
print("🤖 Answer (grounded in live web results):")
print(answer_web)

❓ Question: What are the latest AI safety regulations announced in 2025?

🤖 Answer (grounded in live web results):
In 2025, there were several AI safety regulations announced. 

In the European Union, the EU AI Act officially became law on August 1, 2024, with implementation staggered from early 2025 onwards. The bans on prohibited practices entered the law books in February 2025, and the obligations on general-purpose AI systems like chatbots will start in August 2025. (Source 3 | page web)

In the UK, the AI Opportunities Action Plan was unveiled on January 13, 2025, outlining its commitment to shaping the future of AI and including enabling safe and trusted AI development and adoption through regulation, safety, and assurance. (Source 2 | page web)

In South Korea, the Framework Act on the Development of AI and Establishment of a Foundation of Trust was enacted on January 10th, 2025, a comprehensive AI law sharing common elements with the EU AI Act. (Source 2 | page web)

In the Uni

### D3 — ✏️ YOUR TURN: Compare Local RAG vs Web RAG on the Same Question

Now ask the **same question** to both your local ChromaDB retriever and the web search retriever.  
Choose a question where your local PDF might have partial information but the web would have more current details —  
for example: a topic your PDF covers, but the web has newer developments on.

**Goal:** understand when to use each approach, and when to combine them.

In [37]:
# ✏️ YOUR TURN: pick a question that makes sense for both your local PDF AND the web
# Good examples:
#   - A topic your PDF covers historically, but the web has 2025 updates on
#   - A regulation your PDF mentions, but whose current status you want to verify
#   - A technology your PDF explains, but recent benchmarks exist online

comparison_question = "Is the Great Sword still one of the strongest weapons in Monster Hunter World Iceborne?"

# ── Local RAG (your PDF) ──────────────────────────────────────────────────────
# ✏️ YOUR TURN: use ask_rag() with store_800's retriever
local_retriever = store.as_retriever(search_kwargs={"k": 4})  # YOUR CODE HERE
local_answer, local_docs = ask_rag(comparison_question, local_retriever)  # YOUR CODE HERE

print("=" * 60)
print("📄 LOCAL RAG (from your PDF)")
print("=" * 60)
print("Sources used:", [f"page {d.metadata.get('page') + 1}" if isinstance(d.metadata.get("page"), int) else "page ?" for d in local_docs])
print(local_answer)

print()
print("=" * 60)
print("🌐 WEB RAG (live web search)")
print("=" * 60)

# ✏️ YOUR TURN: use ask_web_rag() with the same question
web_answer = ask_web_rag(comparison_question)  # YOUR CODE HERE
print(web_answer)

📄 LOCAL RAG (from your PDF)
Sources used: ['page 1', 'page 1', 'page 3', 'page 2']
The Great Sword still has a high damage output per swing (page 1) and decent reach (page 1), which are still beneficial in combat. However, its slow movement speed (page 1), slow attack speed (page 1), and long recovery times for attack animations (page 1) are still its weaknesses.

It can still block attacks (page 1) and has charged slashes that can be empowered by charging them (page 2). The Great Sword has 3 kinds of charged slashes: Charged Slash, Strong Charged Slash, and True Charged Slash, each with three different charge levels (page 2).

Overall, while the Great Sword still has its strengths and weaknesses, it is still a viable option in Monster Hunter World Iceborne.

🌐 WEB RAG (live web search)
The Great Sword is still one of the strongest weapons in Monster Hunter World Iceborne, but it requires patience and good positioning to use effectively. According to [Source 1](page web), it's a weapon

### D4 — 🔬 EXPERIMENT: Combining Both (Hybrid Retrieval)

The most powerful approach is to merge results from both your local index AND the web,  
then let the LLM synthesise across all sources. This is how production systems like Perplexity work.

This cell is **fully written** — just run it and observe how the answer changes when the LLM has both local and web context.

In [39]:
HYBRID_SYSTEM_PROMPT = """You are a research assistant with access to both a local document library and live web search results.
You will receive context from two sources: LOCAL DOCUMENT and WEB SEARCH.
Synthesise information from both sources to give the most complete and accurate answer.
Clearly label which source each piece of information comes from:
  - Local: cite the page number
  - Web: cite the URL
If the sources contradict each other, note the discrepancy and explain which seems more current."""

def ask_hybrid_rag(question: str, local_retriever, k_web: int = 3) -> str:
    """Retrieves from both local ChromaDB and web, then generates a combined answer."""
    # Get local chunks
    local_docs = local_retriever.invoke(question)
    # Get web results
    web_docs   = web_search_to_docs(question)

    # Format each source separately so the LLM can attribute correctly
    local_context = format_context(local_docs)
    web_context   = format_context(web_docs)

    prompt = f"""Question: {question}

=== LOCAL DOCUMENT CONTEXT ===
{local_context}

=== LIVE WEB SEARCH CONTEXT ===
{web_context}

Synthesise both sources to answer the question. Label each fact with its source (page number or URL)."""

    client   = Groq(api_key=os.getenv("GROQ_API_KEY"))
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": HYBRID_SYSTEM_PROMPT},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content

# Run with the same question from D3
hybrid_retriever = store_800.as_retriever(search_kwargs={"k": 3})
hybrid_answer    = ask_hybrid_rag(comparison_question, hybrid_retriever)

print("=" * 60)
print("🔀 HYBRID RAG (local PDF + live web)")
print("=" * 60)
print(hybrid_answer)

🔀 HYBRID RAG (local PDF + live web)
Based on the provided sources, here's a synthesis of the information about the Great Sword in Monster Hunter World Iceborne:

The Great Sword is a slow, heavy weapon that sacrifices attack speed for attack strength (LOCAL DOCUMENT, page 1). It has high damage output per swing, decent reach, and can block attacks (LOCAL DOCUMENT, page 1). However, it also has slow movement speed, slow attack speed, and long recovery times for its attack animations (LOCAL DOCUMENT, page 1).

The Great Sword can empower its slashes by charging them, with three different charge levels (LOCAL DOCUMENT, page 2). This allows for a quick squatting motion when the Hunter reaches Level 3, helping with timing the attack (LOCAL DOCUMENT, page 2).

In terms of its effectiveness, the Great Sword is considered a powerful weapon when used correctly (WEB SEARCH, Source 1). It requires patience and positioning to be effective, but can deal massive damage when used properly (WEB SEARCH

### 📝 Reflection D

1. **Local vs Web:** For your chosen question, which source gave a more complete answer?

   *Your answer:* The web source has the more comprehensive and detailed answer.

2. **Hybrid answer:** Did combining both sources produce a noticeably better answer than either alone? What did each source contribute? 
 
   *Your answer:* Yes. The local PDF provided the Great Sword’s main strengths, weaknesses, and mechanics, while the web added tier rankings and extra opinions. Combining both made the answer more detailed and balanced.

3. **When would you choose each approach in a real product?**

   | Scenario | Best approach | Reason |
   |---|---|---|
   | Internal HR policy bot | Local RAG | Policy is private, not on the web |
   | Current news summariser | Web RAG | New changes daily and needs current information |
   | Product manual Q&A with recent updates | Hybrid RAG | The local providdes the official information while the web will give more recent updates |
   | General research assistant | Hybrid RAG | More resources can be found online to add onto the current reasearch paper with trusted documents as well |

4. **What is one risk** of using web-augmented RAG that doesn't exist with local RAG?

   *Your answer:*  Web results may contain inaccurate or unreliable information.

---
## 🧪 PART E — RAG Evaluation with RAGAS  *(Optional — Stretch Goal)*

> **This section is optional.** Complete Parts A–D first. If you have time remaining, come back here.

### Why evaluate?

So far you have been evaluating RAG the hard way: reading retrieved chunks and answers manually and forming a gut feeling about quality. That works for a few questions — but in a real product with thousands of queries, you need **automated, reproducible metrics**.

### What is RAGAS?

**RAGAS** (Retrieval Augmented Generation Assessment) is an open-source Python library that automatically scores your RAG pipeline on four key metrics — using an LLM as a judge. It requires no human-labelled ground truth for the basic metrics.

| Metric | What it asks | Score range |
|---|---|---|
| **Faithfulness** | Is every claim in the answer supported by the retrieved context? (no hallucination) | 0–1 (higher = better) |
| **Answer Relevancy** | Does the answer actually address the user's question? | 0–1 (higher = better) |
| **Context Recall** | Did retrieval find all the evidence needed to answer? | 0–1 (higher = better) |
| **Context Precision** | Are the retrieved chunks relevant? Low = noisy retrieval | 0–1 (higher = better) |

We will run **Faithfulness** and **Answer Relevancy** — the two that don't require ground-truth reference answers, making them the easiest to use.

### E0 — Install RAGAS

In [41]:
import sys
!{sys.executable} -m pip install -q ragas datasets
print("✅ RAGAS installed")

^C
✅ RAGAS installed


### E1 — Guided: Understand the RAGAS Input Format

RAGAS expects a dataset where each row represents one Q&A exchange and contains:

| Field | Type | Description |
|---|---|---|
| `user_input` | `str` | The question that was asked |
| `response` | `str` | The LLM's answer |
| `retrieved_contexts` | `list[str]` | The raw text of each retrieved chunk (as plain strings) |
| `reference` | `str` | *(Optional)* Ground-truth answer — needed for Context Recall but not Faithfulness/Relevancy |

The cell below shows how to build this dataset from your existing `ask_rag()` function.

In [ ]:
# This cell is fully written — run it to see what RAGAS data looks like

# We'll run 3 test questions through our local RAG pipeline
# and collect (question, answer, retrieved_chunks) for each

eval_retriever = store_800.as_retriever(search_kwargs={"k": 4})

# Define a few representative test questions for your document
# 📌 Change these to questions that make sense for YOUR PDF
test_questions = [
    "What are the main topics covered in this document?",
    "What is the current price of Bitcoin?",         # out-of-scope — should score low on faithfulness
    "What does the document say about deadlines?",   # replace with something relevant to your PDF
]

# Collect results
eval_rows = []
for q in test_questions:
    answer, docs = ask_rag(q, eval_retriever)
    eval_rows.append({
        "user_input"          : q,
        "response"            : answer,
        "retrieved_contexts"  : [doc.page_content for doc in docs],  # plain strings, not Document objects
    })
    print(f"✅ Collected: {q[:60]}...")

print(f"\n📊 {len(eval_rows)} rows ready for evaluation")
print()
print("Sample row (first question):")
print(f"  user_input         : {eval_rows[0]['user_input']}")
print(f"  response (first 100): {eval_rows[0]['response'][:100]}...")
print(f"  retrieved_contexts : {len(eval_rows[0]['retrieved_contexts'])} chunks")

### E2 — ✏️ YOUR TURN: Run the RAGAS Evaluation

Now you will build the RAGAS dataset and run evaluation. Fill in the three gaps marked `# ✏️ YOUR TURN`.

**What to expect:**
- The evaluation takes ~30–60 seconds — RAGAS uses an LLM internally to score each answer
- By default RAGAS uses OpenAI. We configure it to use Groq instead (free)
- Scores will be between 0 and 1 — higher is better
- The out-of-scope Bitcoin question should score **low** on faithfulness (the model likely hallucinated)

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# ── Configure RAGAS to use Groq + local embeddings (both free) ─────────────────
ragas_llm = LangchainLLMWrapper(ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY"),
))
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

# ── ✏️ YOUR TURN 1: Build the Hugging Face Dataset from eval_rows ──────────────
# Hint: Dataset.from_list(eval_rows)
eval_dataset = ___________  # YOUR CODE HERE

print(f"Dataset columns : {eval_dataset.column_names}")
print(f"Dataset rows    : {len(eval_dataset)}")
print()

# ── ✏️ YOUR TURN 2: Run the evaluation ─────────────────────────────────────────
# Hint: evaluate(
#     dataset=eval_dataset,
#     metrics=[faithfulness, answer_relevancy],
#     llm=ragas_llm,
#     embeddings=ragas_embeddings,
# )
print("Running RAGAS evaluation (takes ~30–60 seconds)...")
results = ___________  # YOUR CODE HERE

print("\n✅ Evaluation complete!")
print()

# ── ✏️ YOUR TURN 3: Print the results as a readable table ──────────────────────
# Hint: results.to_pandas() gives you a DataFrame
# Print columns: user_input, faithfulness, answer_relevancy
df = ___________  # YOUR CODE HERE — convert results to a DataFrame

# Print per-question scores
print(f"{'Question':<50}  {'Faithful':>9}  {'Relevant':>9}")
print("-" * 72)
for _, row in df.iterrows():
    q_short  = str(row.get('user_input', ''))[:48]
    faith    = row.get('faithfulness',    float('nan'))
    relevant = row.get('answer_relevancy', float('nan'))
    print(f"{q_short:<50}  {faith:>9.3f}  {relevant:>9.3f}")

# Print average scores
print("-" * 72)
print(f"{'AVERAGE':<50}  {df['faithfulness'].mean():>9.3f}  {df['answer_relevancy'].mean():>9.3f}")

### 📝 Reflection E

1. **Faithfulness score for the out-of-scope question** (Bitcoin price): was it low as expected? What does a low faithfulness score indicate?

   *Your answer:*

2. **chunk_size=300 vs chunk_size=800:** did the RAGAS scores change? Which metric changed more — faithfulness or relevancy? Why do you think that is?

   *Your answer:*

3. **Limitation of this evaluation setup:** we are using an LLM (LLaMA via Groq) as both the RAG generator AND the RAGAS judge. What problem could this cause?

   *Your answer:*

4. **In a production RAG system**, how often would you run RAGAS evaluation — and on how many questions?

   *Your answer:*

---
## 🧹 Cleanup

Run this cell when you are done to remove all temporary ChromaDB index folders.

In [1]:
import shutil
from pathlib import Path

dirs_to_clean = [
    "./chroma_lab_notebook",
    "./chroma_exp_300",
    "./chroma_exp_800",
    "./chroma_exp_1500",
    "./chroma_k_exp",
]
for d in dirs_to_clean:
    if Path(d).exists():
        shutil.rmtree(d)
        print(f"🗑  Removed {d}")
print("✅ Cleanup done")

🗑  Removed ./chroma_lab_notebook
🗑  Removed ./chroma_exp_300
🗑  Removed ./chroma_exp_800
🗑  Removed ./chroma_exp_1500
🗑  Removed ./chroma_k_exp
✅ Cleanup done
